# Extension: modality balancing and noise-aware training (Kaggle GPU)

Trains the extension models on the same 44k catalog and split as the main study (same seed, so the split is
identical): gated fusion + OGM gradient modulation (strength tuned on validation), + uni-modal supervision,
+ noise-aware training, + noise-aware and uni-modal. 13 runs, about 1.5 hours on a T4.

**Setup (once)**
1. Verify your phone number on Kaggle (*Settings → Phone verification*); this is needed for GPU and internet.
2. Upload `mmrec_code.zip` as a dataset: *Datasets → New Dataset → upload the zip → name it `mmrec-code` → Create*.
3. In this notebook's right-hand panel: **Accelerator: GPU T4 x2** (or P100), then **Add Input**:
   - **Your Datasets → mmrec-code**
   - search **"Fashion Product Images (Small)"** (by *paramaggarwal*) → add it. This is the 44k-product catalog;
     with it attached **no internet is needed**. (Without it, turn **Internet: On** to download the data instead.)

Upload the **new** `mmrec_code.zip` first (open the mmrec-code dataset → **New Version**).

**Run:** click **Save Version → Save & Run All (Commit) → Save** (about 1.5 hours; you can close the browser).
Afterwards open the version → **Output** → download **`mmrec_extension_results.zip`**.

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "No GPU: right panel > Session options > Accelerator > GPU T4 x2 (or P100)"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Find the project code among the notebook inputs (Kaggle unpacks uploaded zips automatically)
import os, shutil, zipfile, pathlib
PROJ = pathlib.Path("/kaggle/working/mmrec")
if PROJ.exists():
    shutil.rmtree(PROJ)
roots = [pathlib.Path(r) for r in ("/kaggle/input", "/kaggle/working", "/kaggle") if pathlib.Path(r).exists()]
def find(pattern):
    for r in roots:
        yield from r.rglob(pattern)
src_dir = next((p.parent.parent.parent for p in find("experiments.py")
                if p.parent.name == "scratch" and p.parent.parent.name == "src" and PROJ not in p.parents), None)
z = None if src_dir else next(find("mmrec_code.zip"), None)
if src_dir is not None:
    shutil.copytree(src_dir, PROJ)
elif z is not None:
    zipfile.ZipFile(z).extractall(PROJ)
else:
    print("Attached inputs (/kaggle/input):")
    for p in sorted(pathlib.Path("/kaggle/input").glob("*/*"))[:40]:
        print("   ", p)
    raise SystemExit("Project code not found. Right panel > + Add Input > Your Work / Your Datasets > add "
                     "'mmrec-code' (wait until a new version has finished processing), then run again.")
os.chdir(PROJ)
print(sorted(p.name for p in PROJ.iterdir()))

In [ ]:
# 3. Find the product catalog: the attached Kaggle dataset (offline) or a Hugging Face download (needs internet)
catalog = next((p.parent for p in pathlib.Path("/kaggle/input").rglob("styles.csv")), None)
print("Catalog:", catalog or "not attached -> will download from Hugging Face (Internet must be On)")

In [ ]:
# 4. Dataset: full 44k catalog -> 64x64 images, tokenizer, 70/15/15 split
if catalog is not None:
    !python -m src.scratch.data --source folder --path "{catalog}"
else:
    !pip -q install datasets
    !python -m src.scratch.data --source hf
assert pathlib.Path("data/scratch/meta.csv").exists(), (
    "Dataset preparation failed. Attach 'Fashion Product Images (Small)' (Add Input) or turn Internet On.")
import pandas as pd
m = pd.read_csv("data/scratch/meta.csv")
print(len(m), "products,", m.articleType.nunique(), "article types"); print(m.split.value_counts())

In [ ]:
# 5. Extension runs: OGM (alpha tuned on validation), uni-modal supervision, noise-aware training (about 1.5 hours)
OUT = "/kaggle/working/out"
!python -m src.scratch.experiments --out {OUT} --stages balance --base-opt SGD --base-lr 0.1 --bundle-balance /kaggle/working/mmrec_extension_results.zip

In [ ]:
# 6. Keep the Output tab small, then download mmrec_extension_results.zip from the Output tab
for f in ["data/scratch/images.npy", "data/scratch/tokens.npy"]:
    pathlib.Path(f).unlink(missing_ok=True)
print(round(os.path.getsize("/kaggle/working/mmrec_extension_results.zip") / 2**20, 1), "MB")